#### Data Preprocessing & Data Cleaning

In [ ]:
import pandas as pd

# Load semua dataset dari folder 'data',
telemetry = pd.read_csv('PdM_telemetry.csv')
machines = pd.read_csv('PdM_machines.csv')
errors = pd.read_csv('PdM_errors.csv')
maint = pd.read_csv('PdM_maint.csv')
failures = pd.read_csv('PdM_failures.csv')

# Mengecek dimensi dua tabel utama.
print("Dimensi Tabel Machines:", machines.shape)
print("Dimensi Tabel Telemetry:", telemetry.shape)

# Menampilkan 5 baris pertama untuk inspeksi visual.
display(machines.head())
display(telemetry.head())

Dimensi Tabel Machines: (100, 3)
Dimensi Tabel Telemetry: (876100, 6)


,machineID,model,age
0,1,model3,18
1,2,model4,7
2,3,model3,8
3,4,model3,7
4,5,model3,2


,datetime,machineID,volt,rotate,pressure,vibration
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511


#### Data Cleaning

In [ ]:
from IPython.display import display

# Konversi format teks menjadi tipe data Datetime yang benar.
telemetry['datetime'] = pd.to_datetime(telemetry['datetime'])
errors['datetime'] = pd.to_datetime(errors['datetime'])
maint['datetime'] = pd.to_datetime(maint['datetime'])
failures['datetime'] = pd.to_datetime(failures['datetime'])

# Membungkus hasil Cek Data Kosong ke dalam DataFrame (Tabel).
tabel_missing_telemetry = pd.DataFrame(telemetry.isnull().sum(), columns=['Jumlah Data Kosong']).reset_index()
tabel_missing_telemetry.rename(columns={'index': 'Kolom Telemetry'}, inplace=True)

tabel_missing_failures = pd.DataFrame(failures.isnull().sum(), columns=['Jumlah Data Kosong']).reset_index()
tabel_missing_failures.rename(columns={'index': 'Kolom Failures'}, inplace=True)

# Membungkus hasil Rentang Waktu ke dalam DataFrame (Tabel).
tabel_rentang_waktu = pd.DataFrame({
    'Keterangan': ['Data Dimulai Dari', 'Data Berakhir Pada'],
    'Tanggal & Waktu': [telemetry['datetime'].min(), telemetry['datetime'].max()]
})

# Menampilkan hasil dalam bentuk tabel yang rapi.
print("--- Tabel Cek Data Kosong (Telemetry) ---")
display(tabel_missing_telemetry)

print("\n--- Tabel Cek Data Kosong (Failures) ---")
display(tabel_missing_failures)

print("\n--- Tabel Rentang Durasi Operasional Mesin ---")
display(tabel_rentang_waktu)

--- Tabel Cek Data Kosong (Telemetry) ---


,Kolom Telemetry,Jumlah Data Kosong
0,datetime,0
1,machineID,0
2,volt,0
3,rotate,0
4,pressure,0
5,vibration,0



--- Tabel Cek Data Kosong (Failures) ---


,Kolom Failures,Jumlah Data Kosong
0,datetime,0
1,machineID,0
2,failure,0



--- Tabel Rentang Durasi Operasional Mesin ---


,Keterangan,Tanggal & Waktu
0,Data Dimulai Dari,2015-01-01 06:00:00
1,Data Berakhir Pada,2016-01-01 06:00:00


#### Feature Engineering (Meringkas Data) 

In [ ]:
# Meringkas data sensor menjadi rata-rata per 24 jam (Rolling Mean 24h).
telemetry_feat = telemetry.copy()

temp = []
fields = ['volt', 'rotate', 'pressure', 'vibration']

# Melakukan pivot dan menghitung rata-rata 24 jam terakhir untuk setiap mesin.
for col in fields:
    temp.append(pd.pivot_table(telemetry_feat,
                               index='datetime',
                               columns='machineID',
                               values=col).rolling(window=24).mean().unstack())

# Menggabungkan hasil perhitungan.
telemetry_mean_24h = pd.concat(temp, axis=1)
telemetry_mean_24h.columns = [i + '_mean_24h' for i in fields]
telemetry_mean_24h.reset_index(inplace=True)

# Menghapus baris kosong di awal (karena butuh 24 jam pertama untuk mulai menghitung rata-rata).
telemetry_mean_24h = telemetry_mean_24h.dropna()

# Menampilkan hasil.
print("--- Dimensi Data Setelah Diringkas (Rata-rata 24 Jam) ---")
print(telemetry_mean_24h.shape)

print("\n--- 5 Baris Pertama Data Rata-rata Harian ---")
display(telemetry_mean_24h.head())

--- Dimensi Data Setelah Diringkas (Rata-rata 24 Jam) ---
(873800, 6)

--- 5 Baris Pertama Data Rata-rata Harian ---


,machineID,datetime,volt_mean_24h,rotate_mean_24h,pressure_mean_24h,vibration_mean_24h
23,1,2015-01-02 05:00:00,169.733809,445.179865,96.797113,40.385160
24,1,2015-01-02 06:00:00,170.614862,446.364859,96.849785,39.736826
25,1,2015-01-02 07:00:00,170.961598,445.610629,97.179484,39.419434
26,1,2015-01-02 08:00:00,170.525721,443.906847,97.667249,39.786670
27,1,2015-01-02 09:00:00,169.893965,447.009407,97.715600,39.498374


#### Data Merging (Penggabungan Label Target)

In [9]:
# Menggabungkan data sensor (kiri) dengan data kerusakan (kanan).
final_data = pd.merge(telemetry_mean_24h, failures, on=['machineID', 'datetime'], how='left')

# Mengisi baris yang kosong (waktu dimana mesin tidak rusak) dengan teks 'None'.
final_data['failure'] = final_data['failure'].fillna('None')

# Mengecek distribusi jumlah kerusakan mesin.
print("--- Distribusi Status Mesin ---")
display(pd.DataFrame(final_data['failure'].value_counts()).rename(columns={'count': 'Jumlah Kejadian'}))

# Menampilkan contoh baris tepat pada saat mesin rusak.
print("\n--- Contoh Data Saat Mesin Mengalami Kerusakan ---")
display(final_data[final_data['failure'] != 'None'].head())

--- Distribusi Status Mesin ---


,Jumlah Kejadian
failure,
None,873098
comp2,256
comp1,183
comp4,176
comp3,128



--- Contoh Data Saat Mesin Mengalami Kerusakan ---


,machineID,datetime,volt_mean_24h,rotate_mean_24h,pressure_mean_24h,vibration_mean_24h,failure
73,1,2015-01-05 06:00:00,171.929104,443.448775,98.675590,51.780445,comp4
1513,1,2015-03-06 06:00:00,185.130369,444.072120,98.183828,39.788682,comp1
2593,1,2015-04-20 06:00:00,171.274184,373.459705,102.546688,40.081677,comp2
4033,1,2015-06-19 06:00:00,171.098775,461.895240,101.724552,47.957846,comp4
5833,1,2015-09-02 06:00:00,165.256814,442.129535,95.933374,58.047737,comp4
